## 环境准备：安装与导入

下面这些单元主要完成依赖安装、GPU/EGL 渲染配置，以及 MuJoCo / MJX / Brax 的基础导入。

In [ ]:
# 安装 MuJoCo、MJX 和 Brax
# 这是运行整个 notebook 所需的核心依赖。
!pip install mujoco
!pip install mujoco_mjx
!pip install brax

In [ ]:
#@title 检查 MuJoCo 是否安装成功

# 配置 GPU 渲染。
from google.colab import files
import distutils.util
import os
import subprocess
if subprocess.run('nvidia-smi').returncode:
  raise RuntimeError(
      'Cannot communicate with GPU. '
      'Make sure you are using a GPU Colab runtime. '
      'Go to the Runtime menu and select Choose runtime type.')

# 写入 ICD 配置，让 glvnd 能识别 Nvidia 的 EGL 驱动。
# 这部分通常会随 Nvidia 驱动一起安装，但在 Colab 里
# 内核不是通过 APT 安装驱动，因此这里往往缺少对应的 ICD 文件。
# 相关说明可见 libglvnd 的 EGL ICD 枚举文档。
NVIDIA_ICD_CONFIG_PATH = '/usr/share/glvnd/egl_vendor.d/10_nvidia.json'
if not os.path.exists(NVIDIA_ICD_CONFIG_PATH):
  with open(NVIDIA_ICD_CONFIG_PATH, 'w') as f:
    f.write("""{
    "file_format_version" : "1.0.0",
    "ICD" : {
        "library_path" : "libEGL_nvidia.so.0"
    }
}
""")

# 让 MuJoCo 使用 EGL 后端进行渲染（需要 GPU）。
print('Setting environment variable to use GPU rendering:')
%env MUJOCO_GL=egl

# 检查安装是否成功。
try:
  print('Checking that the installation succeeded:')
  import mujoco
  mujoco.MjModel.from_xml_string('<mujoco/>')
except Exception as e:
  raise e from RuntimeError(
      'Something went wrong during installation. Check the shell output above '
      'for more information.\n'
      'If using a hosted Colab runtime, make sure you enable GPU acceleration '
      'by going to the Runtime menu and selecting "Choose runtime type".')

print('Installation successful.')

# 其他辅助导入与工具函数。
import time
import itertools
import numpy as np

# 图形显示与绘图。
print('Installing mediapy:')
!command -v ffmpeg >/dev/null || (apt update && apt install -y ffmpeg)
!pip install -q mediapy
import mediapy as media
import matplotlib.pyplot as plt

# 让 numpy 的打印结果更易读。
np.set_printoptions(precision=3, suppress=True, linewidth=100)

from IPython.display import clear_output
clear_output()

In [ ]:
#@title 导入 MuJoCo、MJX 和 Brax

import os
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = "0.8" # 这里保持原值；0.9 在 Colab 中容易明显变卡。
from datetime import datetime
import functools

# 数学与自动微分相关库
import jax.numpy as jp
import numpy as np
import jax
from jax import config # 解析梯度在双精度下通常更稳定。
config.update("jax_debug_nans", True)
config.update("jax_enable_x64", True)
config.update('jax_default_matmul_precision', 'high')
from brax import math

# 仿真相关
import mujoco
import mujoco.mjx as mjx

# Brax 环境与工具
from brax import envs
from brax.base import Motion, Transform
from brax.io import mjcf
from brax.envs.base import PipelineEnv, State
from brax.mjx.pipeline import _reformat_contact
from brax.training.acme import running_statistics
from brax.io import model

# 训练算法
from brax.training.agents.apg import train as apg
from brax.training.agents.apg import networks as apg_networks
from brax.training.agents.ppo import train as ppo

# 其他支持模块
from etils import epath
import mediapy as media
import matplotlib.pyplot as plt
from ml_collections import config_dict
from typing import Any, Dict


#### 四足机器人环境

这里切换到 **Unitree GO2** 模型，并定义渲染、几何体索引和身体索引等基础工具。

In [ ]:
# 下载 MuJoCo Menagerie，里面包含 GO2 的 MuJoCo 模型文件。
!git clone https://github.com/google-deepmind/mujoco_menagerie.git

In [ ]:
# 载入 GO2 的 XML 模型，并定义后续会用到的 keyframe、足端 geom 名称和髋部 body 名称。
# 这里不再沿用 ANYmal 的命名，而是直接按 GO2 的 XML 结构取索引。

xml_path = epath.Path('mujoco_menagerie/unitree_go2/scene_mjx.xml').as_posix()

mj_model = mujoco.MjModel.from_xml_path(xml_path)

if 'renderer' not in dir():
    renderer = mujoco.Renderer(mj_model)

KEYFRAME_NAME = 'home'
FEET_GEOM_NAMES = ('FL', 'FR', 'RL', 'RR')
HIP_BODY_NAMES = ('FL_hip', 'FR_hip', 'RL_hip', 'RR_hip')

def get_geom_ids(mj_model, geom_names):
    return jp.array([
        mujoco.mj_name2id(mj_model, mujoco.mjtObj.mjOBJ_GEOM, name)
        for name in geom_names
    ])

def get_body_ids(mj_model, body_names):
    return jp.array([
        mujoco.mj_name2id(mj_model, mujoco.mjtObj.mjOBJ_BODY, name) - 1
        for name in body_names
    ])

init_q = mj_model.keyframe(KEYFRAME_NAME).qpos

mj_data = mujoco.MjData(mj_model)
mj_data.qpos = init_q
mujoco.mj_forward(mj_model, mj_data)

renderer.update_scene(mj_data)
media.show_image(renderer.render())


In [ ]:
# 用于把策略 rollout 渲染成视频，便于直观看训练结果。
def render_rollout(reset_fn, step_fn,
                   inference_fn, env,
                   n_steps = 200, camera=None,
                   seed=0):
    rng = jax.random.key(seed)
    render_every = 3
    state = reset_fn(rng)
    rollout = [state.pipeline_state]

    for i in range(n_steps):
        act_rng, rng = jax.random.split(rng)
        ctrl, _ = inference_fn(state.obs, act_rng)
        state = step_fn(state, ctrl)
        if i % render_every == 0:
            rollout.append(state.pipeline_state)

    media.show_video(env.render(rollout, camera=camera),
                     fps=1.0 / (env.dt*render_every),
                     codec='gif')

## 实验一：模仿参考运动学

第一阶段先让 GO2 学会一个稳定的 trot 参考步态，为第二阶段残差学习提供基线策略。

#### 设计参考运动学

这一部分构造一个手写的步态参考轨迹，后面会让策略去模仿它。

In [ ]:
# 这一单元定义 trot 参考轨迹：
# - cos_wave / dcos_wave：生成抬腿轨迹与对应速度
# - make_kinematic_ref：把单腿轨迹拼成四足整机的参考动作

def cos_wave(t, step_period, scale):
    _cos_wave = -jp.cos(((2*jp.pi)/step_period)*t)
    return _cos_wave * (scale/2) + (scale/2)

def dcos_wave(t, step_period, scale):
    """
    余弦波的一阶导数，用来构造参考速度。
    """
    return ((scale*jp.pi) / step_period) * jp.sin(((2*jp.pi)/step_period)*t)

def make_kinematic_ref(sinusoid, step_k, scale=0.3, dt=1/50):
    """
    为 12 个关节构造 trot 参考运动学。
    step_k 表示一条腿完成一次抬起和放下所需的时间步数。
    一个完整步态周期长度为 2 * step_k * dt 秒。
    """

    _steps = jp.arange(step_k)
    step_period = step_k * dt
    t = _steps * dt

    wave = sinusoid(t, step_period, scale)
    # 构造“前腿处于摆动相”时的一段关节参考动作
    fleg_cmd_block = jp.concatenate(
        [jp.zeros((step_k, 1)),
        wave.reshape(step_k, 1),
        -2*wave.reshape(step_k, 1)],
        axis=1
    )
    # 在 GO2 上，后腿这里沿用与前腿一致的关节空间参考模式。
    h_leg_cmd_block = fleg_cmd_block

    block1 = jp.concatenate([
        jp.zeros((step_k, 3)),
        fleg_cmd_block,
        h_leg_cmd_block,
        jp.zeros((step_k, 3))],
        axis=1
    )

    block2 = jp.concatenate([
        fleg_cmd_block,
        jp.zeros((step_k, 3)),
        jp.zeros((step_k, 3)),
        h_leg_cmd_block],
        axis=1
    )
    # 一个完整步态周期中，两组对角腿都会经历支撑相与摆动相
    step_cycle = jp.concatenate([block1, block2], axis=0)
    return step_cycle


In [ ]:
# 先把构造出来的参考轨迹可视化，确认参考动作的时序是否合理。
poses  = make_kinematic_ref(cos_wave, step_k=25)

frames = []
init_q = mj_model.keyframe(KEYFRAME_NAME).qpos
mj_data.qpos = init_q
default_ap = init_q[7:]

for i in range(len(poses)):
    mj_data.qpos[7:] = poses[i] + default_ap
    mujoco.mj_forward(mj_model, mj_data)
    renderer.update_scene(mj_data)
    frames.append(renderer.render())

media.show_video(frames, fps=50, codec='gif')


#### 强化学习环境

这里定义第一阶段的训练环境 `TrotGo2`，目标是学习参考步态。

In [ ]:
# 这一单元定义第一阶段模仿学习环境、观测、终止条件和奖励函数。

def get_config():
  def get_default_rewards_config():
    default_config = config_dict.ConfigDict(
        dict(
            scales=config_dict.ConfigDict(
              dict(
                min_reference_tracking = -2.5 * 3e-3, # 用来平衡不同奖励项的量级
                reference_tracking = -1.0,
                feet_height = -1.0
                )
              )
            )
    )
    return default_config

  default_config = config_dict.ConfigDict(
      dict(rewards=get_default_rewards_config(),))

  return default_config

# 下面这些数学函数改编自 diffmimic，用于姿态表示转换。
def quaternion_to_matrix(quaternions):
    r, i, j, k = quaternions[..., 0], quaternions[..., 1], quaternions[..., 2], quaternions[..., 3]
    two_s = 2.0 / (quaternions * quaternions).sum(-1)

    o = jp.stack(
        (
            1 - two_s * (j * j + k * k),
            two_s * (i * j - k * r),
            two_s * (i * k + j * r),
            two_s * (i * j + k * r),
            1 - two_s * (i * i + k * k),
            two_s * (j * k - i * r),
            two_s * (i * k - j * r),
            two_s * (j * k + i * r),
            1 - two_s * (i * i + j * j),
        ),
        -1,
    )
    return o.reshape(quaternions.shape[:-1] + (3, 3))

def matrix_to_rotation_6d(matrix):
    batch_dim = matrix.shape[:-2]
    return matrix[..., :2, :].reshape(batch_dim + (6,))

def quaternion_to_rotation_6d(quaternion):
    return matrix_to_rotation_6d(quaternion_to_matrix(quaternion))

# 第一阶段环境：让 GO2 学习模仿参考 trot 步态。
class TrotGo2(PipelineEnv):

  def __init__(
      self,
      termination_height: float=0.25,
      **kwargs,
  ):
    step_k = kwargs.pop('step_k', 25)

    physics_steps_per_control_step = 10
    kwargs['n_frames'] = kwargs.get(
        'n_frames', physics_steps_per_control_step)

    mj_model = mujoco.MjModel.from_xml_path(xml_path)
    kp = 230
    mj_model.actuator_gainprm[:, 0] = kp
    mj_model.actuator_biasprm[:, 1] = -kp

    sys = mjcf.load_model(mj_model)

    super().__init__(sys=sys, **kwargs)

    self.termination_height = termination_height

    self._init_q = mj_model.keyframe(KEYFRAME_NAME).qpos

    self.err_threshold = 0.4 # 继承 diffmimic 论文中的阈值设定。

    self._default_ap_pose = mj_model.keyframe(KEYFRAME_NAME).qpos[7:]
    self.reward_config = get_config()

    self.action_loc = self._default_ap_pose
    self.action_scale = jp.array([0.2, 0.8, 0.8] * 4)

    self.feet_inds = get_geom_ids(mj_model, FEET_GEOM_NAMES)

    #### 模仿学习所使用的参考轨迹 ####
    kinematic_ref_qpos = make_kinematic_ref(
      cos_wave, step_k, scale=0.3, dt=self.dt)
    kinematic_ref_qvel = make_kinematic_ref(
      dcos_wave, step_k, scale=0.3, dt=self.dt)

    self.l_cycle = jp.array(kinematic_ref_qpos.shape[0])

    # 把关节参考轨迹扩展回完整状态空间（含 base 状态）。

    kinematic_ref_qpos += self._default_ap_pose
    ref_qs = np.tile(self._init_q.reshape(1, 19), (self.l_cycle, 1))
    ref_qs[:, 7:] = kinematic_ref_qpos
    self.kinematic_ref_qpos = jp.array(ref_qs)

    ref_qvels = np.zeros((self.l_cycle, 18))
    ref_qvels[:, 6:] = kinematic_ref_qvel
    self.kinematic_ref_qvel = jp.array(ref_qvels)

    # 使用 checkpoint 可明显降低 JIT 和训练的墙钟时间开销。
    self.pipeline_step = jax.checkpoint(self.pipeline_step,
      policy=jax.checkpoint_policies.dots_with_no_batch_dims_saveable)

  def reset(self, rng: jax.Array) -> State:
    # 使用确定性初始状态进行 reset

    qpos = jp.array(self._init_q)
    qvel = jp.zeros(18)

    data = self.pipeline_init(qpos, qvel)

    # 通过接触信息把机器人精确放到地面上
    pen = jp.min(data.contact.dist)
    qpos = qpos.at[2].set(qpos[2] - pen)
    data = self.pipeline_init(qpos, qvel)

    state_info = {
        'rng': rng,
        'steps': 0.0,
        'reward_tuple': {
            'reference_tracking': 0.0,
            'min_reference_tracking': 0.0,
            'feet_height': 0.0
        },
        'last_action': jp.zeros(12), # 延续 MJX tutorial 的做法，把上一步动作纳入状态。
        'kinematic_ref': jp.zeros(19),
    }

    x, xd = data.x, data.xd
    obs = self._get_obs(data.qpos, x, xd, state_info)
    reward, done = jp.zeros(2)
    metrics = {}
    for k in state_info['reward_tuple']:
      metrics[k] = state_info['reward_tuple'][k]
    state = State(data, obs, reward, done, metrics, state_info)
    return jax.lax.stop_gradient(state)

  def step(self, state: State, action: jax.Array) -> State:
    action = jp.clip(action, -1, 1) # 先裁剪原始动作，避免越界。

    action = self.action_loc + (action * self.action_scale)

    data = self.pipeline_step(state.pipeline_state, action)

    ref_qpos = self.kinematic_ref_qpos[jp.array(state.info['steps']%self.l_cycle, int)]
    ref_qvel = self.kinematic_ref_qvel[jp.array(state.info['steps']%self.l_cycle, int)]

    # 计算参考状态在最大坐标系下对应的运动学量
    ref_data = data.replace(qpos=ref_qpos, qvel=ref_qvel)
    ref_data = mjx.forward(self.sys, ref_data)
    ref_x, ref_xd = ref_data.x, ref_data.xd

    state.info['kinematic_ref'] = ref_qpos

    # 当前时刻的观测量
    x, xd = data.x, data.xd
    obs = self._get_obs(data.qpos, x, xd, state.info)

    # 如果翻倒或高度过低，则终止当前 episode。
    done = 0.0
    done = jp.where(x.pos[0, 2] < self.termination_height, 1.0, done)
    up = jp.array([0.0, 0.0, 1.0])
    done = jp.where(jp.dot(math.rotate(up, x.rot[0]), up) < 0, 1.0, done)

    # 计算奖励
    reward_tuple = {
        'reference_tracking': (
          self._reward_reference_tracking(x, xd, ref_x, ref_xd)
          * self.reward_config.rewards.scales.reference_tracking
        ),
        'min_reference_tracking': (
          self._reward_min_reference_tracking(ref_qpos, ref_qvel, state)
          * self.reward_config.rewards.scales.min_reference_tracking
        ),
        'feet_height': (
          self._reward_feet_height(data.geom_xpos[self.feet_inds][:, 2]
                                   ,ref_data.geom_xpos[self.feet_inds][:, 2])
          * self.reward_config.rewards.scales.feet_height
        )
    }

    reward = sum(reward_tuple.values())

    # 更新状态缓存与统计量
    state.info['reward_tuple'] = reward_tuple
    state.info['last_action'] = action # 保存上一步动作，供下一步观测使用。

    for k in state.info['reward_tuple'].keys():
      state.metrics[k] = state.info['reward_tuple'][k]

    state = state.replace(
        pipeline_state=data, obs=obs, reward=reward,
        done=done)

    #### 如果状态偏离参考太远，则把状态拉回参考附近 ####
    error = (((x.pos - ref_x.pos) ** 2).sum(-1)**0.5).mean()
    to_reference = jp.where(error > self.err_threshold, 1.0, 0.0)

    to_reference = jp.array(to_reference, dtype=int) # 转成 int，保持 JAX 分支输出类型一致。
    ref_data = self.mjx_to_brax(ref_data)

    data = jax.tree_util.tree_map(lambda x, y:
                                  jp.array((1-to_reference)*x + to_reference*y, x.dtype), data, ref_data)

    x, xd = data.x, data.xd # 如果前面重置到了参考状态，这里的 data 可能已更新。
    obs = self._get_obs(data.qpos, x, xd, state.info)

    return state.replace(pipeline_state=data, obs=obs)

  def _get_obs(self, qpos: jax.Array, x: Transform, xd: Motion,
               state_info: Dict[str, Any]) -> jax.Array:

    inv_base_orientation = math.quat_inv(x.rot[0])
    local_rpyrate = math.rotate(xd.ang[0], inv_base_orientation)

    obs_list = []
    # 偏航角速度
    obs_list.append(jp.array([local_rpyrate[2]]) * 0.25)
    # 重力方向在机体坐标系下的投影
    obs_list.append(
        math.rotate(jp.array([0.0, 0.0, -1.0]), inv_base_orientation))
    # 电机关节角
    angles = qpos[7:19]
    obs_list.append(angles - self._default_ap_pose)
    # 上一步动作
    obs_list.append(state_info['last_action'])
    # 当前时刻对应的参考运动学状态
    kin_ref = self.kinematic_ref_qpos[jp.array(state_info['steps']%self.l_cycle, int)]
    obs_list.append(kin_ref[7:]) # 前 7 维对应 base 状态，这里只拼接关节部分。

    obs = jp.clip(jp.concatenate(obs_list), -100.0, 100.0)

    return obs

  def mjx_to_brax(self, data):
    """
    Apply the brax wrapper on the core MJX data structure.
    """
    q, qd = data.qpos, data.qvel
    x = Transform(pos=data.xpos[1:], rot=data.xquat[1:])
    cvel = Motion(vel=data.cvel[1:, 3:], ang=data.cvel[1:, :3])
    offset = data.xpos[1:, :] - data.subtree_com[self.sys.body_rootid[1:]]
    offset = Transform.create(pos=offset)
    xd = offset.vmap().do(cvel)
    data = _reformat_contact(self.sys, data)
    return data.replace(q=q, qd=qd, x=x, xd=xd)


  # ------------ 奖励函数 ----------------
  def _reward_reference_tracking(self, x, xd, ref_x, ref_xd):
    """
    Rewards based on inertial-frame body positions.
    Notably, we use a high-dimension representation of orientation.
    """

    f = lambda x, y: ((x - y) ** 2).sum(-1).mean()

    _mse_pos = f(x.pos,  ref_x.pos)
    _mse_rot = f(quaternion_to_rotation_6d(x.rot),
                 quaternion_to_rotation_6d(ref_x.rot))
    _mse_vel = f(xd.vel, ref_xd.vel)
    _mse_ang = f(xd.ang, ref_xd.ang)

    # 调到和其他奖励项的数量级大致一致。
    return _mse_pos      \
      + 0.1 * _mse_rot   \
      + 0.01 * _mse_vel  \
      + 0.001 * _mse_ang

  def _reward_min_reference_tracking(self, ref_qpos, ref_qvel, state):
    """
    Using minimal coordinates. Improves accuracy of joint angle tracking.
    """
    pos = jp.concatenate([
      state.pipeline_state.qpos[:3],
      state.pipeline_state.qpos[7:]])
    pos_targ = jp.concatenate([
      ref_qpos[:3],
      ref_qpos[7:]])
    pos_err = jp.linalg.norm(pos_targ - pos)
    vel_err = jp.linalg.norm(state.pipeline_state.qvel- ref_qvel)

    return pos_err + vel_err

  def _reward_feet_height(self, feet_pos, feet_pos_ref):
    return jp.sum(jp.abs(feet_pos - feet_pos_ref)) # 用 L1 范数约束足端误差尽量接近 0。

envs.register_environment('trotting_go2', TrotGo2)


#### 使用 FoPG / APG 进行模仿学习

保持原来的训练超参数，只训练第一阶段的 trot 策略。

In [ ]:
# 第一阶段 trot 模仿学习的训练配置。
# 这里保持原有超参数，不额外改动。

make_networks_factory = functools.partial(
    apg_networks.make_apg_networks,
    hidden_layer_sizes=(256, 128)
)

epochs = 499

train_fn = functools.partial(apg.train,
                             episode_length=240,
                             policy_updates=epochs,
                             horizon_length=32,
                             num_envs=64,
                             learning_rate=1e-4,
                             num_eval_envs=64,
                             num_evals=10 + 1,
                             use_float64=True,
                             normalize_observations=True,
                             network_factory=make_networks_factory)

In [ ]:
# 开始第一阶段训练，并记录评估曲线。

x_data = []
y_data = []
ydataerr = []
times = [datetime.now()]

def progress(it, metrics):
  times.append(datetime.now())
  x_data.append(it)
  y_data.append(metrics['eval/episode_reward'])
  ydataerr.append(metrics['eval/episode_reward_std'])

# 通过 monkey patch 修复 mjx_to_brax 的兼容性问题。
from brax.base import Motion, Transform
def fixed_mjx_to_brax(self, data):
  q, qd = data.qpos, data.qvel
  x = Transform(pos=data.xpos[1:], rot=data.xquat[1:])
  cvel = Motion(vel=data.cvel[1:, 3:], ang=data.cvel[1:, :3])
  offset = data.xpos[1:, :] - data.subtree_com[self.sys.body_rootid[1:]]
  offset = Transform.create(pos=offset)
  xd = offset.vmap().do(cvel)
  # 这里先忽略 contact 的重格式化，避免触发 AttributeError。
  # contact = _reformat_contact(self.sys, data.contact)
  return data.replace(q=q, qd=qd, x=x, xd=xd)

TrotGo2.mjx_to_brax = fixed_mjx_to_brax

# 这里设置 step_k=13，对应每只脚大约每秒两次着地。
env = envs.get_environment("trotting_go2", step_k = 13)
eval_env = envs.get_environment("trotting_go2", step_k = 13)

make_inference_fn, params, _= train_fn(environment=env,
                                       progress_fn=progress,
                                       eval_env=eval_env)

plt.errorbar(x_data, y_data, yerr=ydataerr)


In [ ]:
# 渲染第一阶段学到的 trot 策略，并把参数保存到本地路径。

demo_env = envs.training.EpisodeWrapper(env,
                                        episode_length=1000,
                                        action_repeat=1)

render_rollout(
  jax.jit(demo_env.reset),
  jax.jit(demo_env.step),
  jax.jit(make_inference_fn(params)),
  demo_env,
  n_steps=200,
  seed=1
)

model_path = '/tmp/trotting_2hz_policy'
model.save_params(model_path, params)
print(f"第一阶段策略已保存到: {model_path}")


## 实验二：四足前向运动

第二阶段在第一阶段 trot 策略的基础上做残差学习，让 GO2 学会稳定向前运动。

#### 强化学习环境

这里定义第二阶段的前向运动环境 `FwdTrotGo2`。

In [ ]:
# 这一单元定义第二阶段前向运动环境、残差学习观测和奖励函数。

def axis_angle_to_quaternion(v: jp.ndarray, theta:jp.float_):
    """
    轴角表示：绕单位向量 v 旋转 theta。
    """
    return jp.concatenate([jp.cos(0.5*theta).reshape(1), jp.sin(0.5*theta)*v.reshape(3)])

def get_config():
  """返回 GO2 前向运动环境的奖励配置。"""

  def get_default_rewards_config():
    default_config = config_dict.ConfigDict(
        dict(
            scales=config_dict.ConfigDict(
                dict(
                    tracking_lin_vel = 1.0,
                    orientation = -1.0, # 惩罚机体底座不够平。
                    height = 0.5,
                    lin_vel_z=-1.0, # 防止学出“腾空/扑倒”这类投机策略。
                    torque = -0.01,
                    feet_pos = -1, # 约束足端位置，不让动作走向硬编码极端。
                    feet_height = -1, # 防止策略只站着不动。
                    joint_velocity = -0.001
                    )
            ),
        )
    )
    return default_config

  default_config = config_dict.ConfigDict(
      dict(rewards=get_default_rewards_config(),))

  return default_config

# 第二阶段环境：在 trot baseline 上做残差学习，得到前向运动策略。
class FwdTrotGo2(PipelineEnv):

  def __init__(
      self,
      termination_height: float=0.25,
      **kwargs,
  ):

    self.target_vel = kwargs.pop('target_vel', 0.75)
    step_k = kwargs.pop('step_k', 25)
    self.baseline_inference_fn = kwargs.pop("baseline_inference_fn")
    physics_steps_per_control_step = 10
    kwargs['n_frames'] = kwargs.get(
        'n_frames', physics_steps_per_control_step)
    self.termination_height = termination_height

    mj_model = mujoco.MjModel.from_xml_path(xml_path)
    kp = 230
    mj_model.actuator_gainprm[:, 0] = kp
    mj_model.actuator_biasprm[:, 1] = -kp
    self._init_q = mj_model.keyframe(KEYFRAME_NAME).qpos
    self._default_ap_pose = mj_model.keyframe(KEYFRAME_NAME).qpos[7:]
    self.reward_config = get_config()

    self.action_loc = self._default_ap_pose
    self.action_scale = jp.array([0.2, 0.8, 0.8] * 4)

    self.target_h = self._init_q[2]

    sys = mjcf.load_model(mj_model)
    super().__init__(sys=sys, **kwargs)

    """
    参考运动学只用于步态调度，不直接作为最终动作。
    """

    kinematic_ref_qpos = make_kinematic_ref(
      cos_wave, step_k, scale=0.3, dt=self.dt)
    self.l_cycle = jp.array(kinematic_ref_qpos.shape[0])
    self.kinematic_ref_qpos = jp.array(kinematic_ref_qpos + self._default_ap_pose)

    """
    足端跟踪相关变量。
    """
    gait_k = step_k * 2
    self.gait_period = gait_k * self.dt

    self.step_k = step_k
    self.feet_inds = get_geom_ids(mj_model, FEET_GEOM_NAMES)
    self.hip_inds = get_body_ids(mj_model, HIP_BODY_NAMES)

    self.pipeline_step = jax.checkpoint(self.pipeline_step,
      policy=jax.checkpoint_policies.dots_with_no_batch_dims_saveable)

  def reset(self, rng: jax.Array) -> State:
    rng, key_xyz, key_ang, key_ax, key_q, key_qd = jax.random.split(rng, 6)

    qpos = jp.array(self._init_q)
    qvel = jp.zeros(18)

    #### 给初始状态加入随机扰动，提高策略鲁棒性 ####

    r_xyz = 0.2 * (jax.random.uniform(key_xyz, (3,))-0.5)
    r_angle = (jp.pi/12) * (jax.random.uniform(key_ang, (1,)) - 0.5) # 约 15 度范围内的姿态扰动。
    r_axis = (jax.random.uniform(key_ax, (3,)) - 0.5)
    r_axis = r_axis / jp.linalg.norm(r_axis)
    r_quat = axis_angle_to_quaternion(r_axis, r_angle)

    r_joint_q = 0.2 * (jax.random.uniform(key_q, (12,)) - 0.5)
    r_joint_qd = 0.1 * (jax.random.uniform(key_qd, (12,)) - 0.5)

    qpos = qpos.at[0:3].set(qpos[0:3] + r_xyz)
    qpos = qpos.at[3:7].set(r_quat)
    qpos = qpos.at[7:19].set(qpos[7:19] + r_joint_q)
    qvel = qvel.at[6:18].set(qvel[6:18] + r_joint_qd)

    data = self.pipeline_init(qpos, qvel)

    # 确保 reset 后机器人既不会陷入地面，也不会悬空。
    pen = jp.min(data.contact.dist)
    qpos = qpos.at[2].set(qpos[2] - pen)
    data = self.pipeline_init(qpos, qvel)

    state_info = {
        'rng': rng,
        'steps': 0.0,
        'reward_tuple': {
            'tracking_lin_vel': 0.0,
            'orientation': 0.0,
            'height': 0.0,
            'lin_vel_z': 0.0,
            'torque': 0.0,
            'joint_velocity': 0.0,
            'feet_pos': 0.0,
            'feet_height': 0.0
        },
        'last_action': jp.zeros(12), # 延续 MJX tutorial 的做法，把上一步动作纳入状态。
        'baseline_action': jp.zeros(12),
        'xy0': jp.zeros((4, 2)),
        'k0': 0.0,
        'xy*': jp.zeros((4, 2))
    }

    x, xd = data.x, data.xd
    _obs = self._get_obs(data.qpos, x, xd, state_info) # 先构造内部观测，再交给 baseline trotter。

    action_key, key = jax.random.split(state_info['rng'])
    state_info['rng'] = key
    next_action, _ = self.baseline_inference_fn(_obs, action_key)

    obs = jp.concatenate([_obs, next_action])

    reward, done = jp.zeros(2)
    metrics = {}
    for k in state_info['reward_tuple']:
      metrics[k] = state_info['reward_tuple'][k]
    state = State(data, obs, reward, done, metrics, state_info)
    return jax.lax.stop_gradient(state)

  def step(self, state: State, action: jax.Array) -> State:

    action = jp.clip(action, -1, 1)

    cur_base = state.obs[-12:]
    action += cur_base
    state.info['baseline_action'] = cur_base

    action = self.action_loc + (action * self.action_scale)

    data = self.pipeline_step(state.pipeline_state, action)

    # 当前时刻的观测量
    x, xd = data.x, data.xd
    obs = self._get_obs(data.qpos, x, xd, state.info)

    # 如果翻倒或高度过低，则终止当前 episode。
    done = 0.0
    done = jp.where(x.pos[0, 2] < self.termination_height, 1.0, done)
    up = jp.array([0.0, 0.0, 1.0])
    done = jp.where(jp.dot(math.rotate(up, x.rot[0]), up) < 0, 1.0, done)

    #### 更新足端参考落点 ####

    # 检测是否进入了新的步态半周期
    s = state.info['steps']
    step_num = s // (self.step_k)
    even_step = step_num % 2 == 0
    new_step = (s % self.step_k) == 0
    new_even_step = jp.logical_and(new_step, even_step)
    new_odd_step = jp.logical_and(new_step, jp.logical_not(even_step))

    # 用 Raibert heuristic 计算下一次落脚的目标位置
    hip_xy = x.pos[self.hip_inds][:,:2] # 4 x 2，四条腿各自髋部在平面内的位置。
    v_body = data.qvel[0:2]
    step_period = self.gait_period/2
    raibert_xy = hip_xy + (step_period/2) * v_body

    # 更新缓存中的参考足端位置。
    cur_tars = state.info['xy*']
    i_FRRL = jp.array([1, 2])
    i_FLRR = jp.array([0, 3])
    feet_xy = data.geom_xpos[self.feet_inds][:,:2]

    # 对于 trot 步态，每次只更新一组对角腿，
    # 另一组保持相对固定。
    case_c1 = raibert_xy.at[i_FLRR].set(feet_xy[i_FLRR])
    case_c2 = raibert_xy.at[i_FRRL].set(feet_xy[i_FRRL])
    xy_tars = jp.where(new_even_step, case_c1, cur_tars)
    xy_tars = jp.where(new_odd_step, case_c2, xy_tars)
    state.info['xy*'] = xy_tars

    # 记录新步态阶段开始时的时间与足端位置。
    state.info['k0'] = jp.where(new_step,
                                state.info['steps'],
                                state.info['k0'])
    state.info['xy0'] = jp.where(new_step,
                                 feet_xy,
                                 state.info['xy0'])

    # 计算奖励
    reward_tuple = {
        'tracking_lin_vel': (
            self._reward_tracking_lin_vel(jp.array([self.target_vel, 0, 0]), x, xd)
            * self.reward_config.rewards.scales.tracking_lin_vel
        ),
        'orientation': (
          self._reward_orientation(x)
          * self.reward_config.rewards.scales.orientation
        ),
        'lin_vel_z': (
            self._reward_lin_vel_z(xd)
            * self.reward_config.rewards.scales.lin_vel_z
        ),
        'height': (
          self._reward_height(data.qpos)
          * self.reward_config.rewards.scales.height
        ),
        'torque': (
          self._reward_action(data.qfrc_actuator)
          * self.reward_config.rewards.scales.torque
        ),
        'joint_velocity': (
          self._reward_joint_velocity(data.qvel)
          * self.reward_config.rewards.scales.joint_velocity
        ),
        'feet_pos': (
          self._reward_feet_pos(data, state)
          * self.reward_config.rewards.scales.feet_pos
        ),
        'feet_height': (
          self._reward_feet_height(data, state.info)
          * self.reward_config.rewards.scales.feet_height
        )
    }

    reward = sum(reward_tuple.values())

    # 更新状态缓存与统计量
    state.info['reward_tuple'] = reward_tuple
    state.info['last_action'] = action

    for k in state.info['reward_tuple'].keys():
      state.metrics[k] = state.info['reward_tuple'][k]

    # 先用 baseline 策略给出下一步动作，再和残差环境观测拼接
    action_key, key = jax.random.split(state.info['rng'])
    state.info['rng'] = key
    next_action, _ = self.baseline_inference_fn(obs, action_key)
    obs = jp.concatenate([obs, next_action])

    state = state.replace(
        pipeline_state=data, obs=obs, reward=reward,
        done=done)
    return state

  def _get_obs(self, qpos: jax.Array, x: Transform, xd: Motion,
               state_info: Dict[str, Any]) -> jax.Array:

    inv_base_orientation = math.quat_inv(x.rot[0])
    local_rpyrate = math.rotate(xd.ang[0], inv_base_orientation)

    obs_list = []
    # 偏航角速度
    obs_list.append(jp.array([local_rpyrate[2]]) * 0.25)
    # 重力方向在机体坐标系下的投影
    obs_list.append(
        math.rotate(jp.array([0.0, 0.0, -1.0]), inv_base_orientation))
    # 电机关节角
    angles = qpos[7:19]
    obs_list.append(angles - self._default_ap_pose)
    # 上一步动作
    obs_list.append(state_info['last_action'])
    # 步态相位/调度信息
    kin_ref = self.kinematic_ref_qpos[jp.array(state_info['steps']%self.l_cycle, int)]
    obs_list.append(kin_ref)

    obs = jp.clip(jp.concatenate(obs_list), -100.0, 100.0)

    return obs

  # ------------ 奖励函数 ----------------
  def _reward_tracking_lin_vel(
      self, commands: jax.Array, x: Transform, xd: Motion) -> jax.Array:
    # 跟踪平面线速度目标
    local_vel = math.rotate(xd.vel[0], math.quat_inv(x.rot[0]))
    lin_vel_error = jp.sum(jp.square(commands[:2] - local_vel[:2]))
    lin_vel_reward = jp.exp(-lin_vel_error)
    return lin_vel_reward
  def _reward_orientation(self, x: Transform) -> jax.Array:
    # 惩罚机体不够水平
    up = jp.array([0.0, 0.0, 1.0])
    rot_up = math.rotate(up, x.rot[0])
    return jp.sum(jp.square(rot_up[:2]))
  def _reward_lin_vel_z(self, xd: Motion) -> jax.Array:
    # 惩罚 z 方向速度过大
    return jp.clip(jp.square(xd.vel[0, 2]), 0, 10)
  def _reward_joint_velocity(self, qvel):
      return jp.clip(jp.sqrt(jp.sum(jp.square(qvel[6:]))), 0, 100)
  def _reward_height(self, qpos) -> jax.Array:
    return jp.exp(-jp.abs(qpos[2] - self.target_h)) # 希望机体高度接近目标高度。
  def _reward_action(self, action) -> jax.Array:
    return jp.sqrt(jp.sum(jp.square(action)))
  def _reward_feet_pos(self, data, state):
    dt = (state.info['steps'] - state.info['k0']) * self.dt # 标量，表示当前步态阶段内已走过的时间。
    step_period = self.gait_period / 2
    xyt = state.info['xy0'] + (state.info['xy*'] - state.info['xy0']) * (dt/step_period)

    feet_pos = data.geom_xpos[self.feet_inds][:, :2]

    rews = jp.sum(jp.square(feet_pos - xyt), axis=1)
    rews = jp.clip(rews, 0, 10)
    return jp.sum(rews)
  def _reward_feet_height(self, data, state_info):
    """
    Feet height tracks rectified sine waves
    """
    h_tar = 0.1
    t = state_info['steps'] * self.dt
    offset = self.gait_period/2
    ref1 = jp.sin((2*jp.pi/self.gait_period)*t) # FR 和 RL 两条腿的参考相位。
    ref2 = jp.sin((2*jp.pi/self.gait_period)*(t - offset)) # FL 和 RR 两条腿的参考相位。

    ref1, ref2 = ref1 * h_tar, ref2 * h_tar
    h_tars = jp.array([ref2, ref1, ref1, ref2])
    h_tars = h_tars.clip(min=0, max=None) + 0.02 # 给摆动相足端一个小的抬脚高度偏置。

    feet_height = data.geom_xpos[self.feet_inds][:,2]
    errs = jp.clip(jp.square(feet_height - h_tars), 0, 10)
    return jp.sum(errs)

envs.register_environment('go2', FwdTrotGo2)


#### 使用 FoPG / APG 进行残差学习

这一阶段会先重建第一阶段的推理函数，再加载 trot 策略参数作为 baseline。

In [ ]:
# 第二阶段训练前，需要先加载第一阶段的 trot baseline。

# 先重建第一阶段 trot 策略的推理函数。
make_networks_factory = functools.partial(
    apg_networks.make_apg_networks,
    hidden_layer_sizes=(256, 128)
)

nets = make_networks_factory(observation_size=1, # 这里只是为了参数初始化，占位即可。
                             action_size=12,
                             preprocess_observations_fn=running_statistics.normalize)

make_inference_fn = apg_networks.make_inference_fn(nets)

# 再配置第二阶段前向运动训练。
make_networks_factory = functools.partial(
    apg_networks.make_apg_networks,
    hidden_layer_sizes=(128, 64)
)

epochs = 499

train_fn = functools.partial(apg.train,
                             episode_length=1000,
                             policy_updates=epochs,
                             horizon_length=32,
                             num_envs=64,
                             learning_rate=1.5e-4,
                             schedule_decay=0.995,
                             num_eval_envs=64,
                             num_evals=10 + 1,
                             use_float64=True,
                             normalize_observations=True,
                             network_factory=make_networks_factory)

model_path = '/tmp/trotting_2hz_policy'
params = model.load_params(model_path)
baseline_inference_fn = make_inference_fn(params)

env_kwargs = dict(target_vel=0.75, step_k=13,
                  baseline_inference_fn=baseline_inference_fn)

In [ ]:
# 开始第二阶段训练，并记录评估曲线。

x_data = []
y_data = []
ydataerr = []
times = [datetime.now()]

def progress(it, metrics):
  times.append(datetime.now())
  x_data.append(it)
  y_data.append(metrics['eval/episode_reward'])
  ydataerr.append(metrics['eval/episode_reward_std'])

env = envs.get_environment("go2", **env_kwargs)
eval_env = envs.get_environment("go2", **env_kwargs)

make_inference_fn, params, _= train_fn(environment=env,
                                       progress_fn=progress,
                                       eval_env=eval_env)

plt.errorbar(x_data, y_data, yerr=ydataerr)


In [ ]:
# 渲染第二阶段学到的前向运动策略。

demo_env = envs.training.EpisodeWrapper(env,
                                        episode_length=1000,
                                        action_repeat=1)

render_rollout(
  jax.jit(demo_env.reset),
  jax.jit(demo_env.step),
  jax.jit(make_inference_fn(params)),
  demo_env,
  n_steps=200,
  camera="track"
)

In [ ]:
# 保存第二阶段最终前向运动策略参数。

model_path_forward = '/tmp/forward_locomotion_policy'
model.save_params(model_path_forward, params)
print(f"第二阶段前向运动策略已保存到: {model_path_forward}")

#### 训练结果导出到本地（Colab 下载）

下面这个单元不会改动训练逻辑，只是把已经保存好的策略文件整理并打包，然后直接下载到你自己的电脑。

In [ ]:
# 把训练得到的策略文件打包成 zip，并通过 Colab 下载到本地。
# 说明：这里同时导出第一阶段 trot 策略和第二阶段前向运动策略。

from google.colab import files
import os
import shutil

EXPORT_DIR = '/content/go2_policy_export'
ZIP_PATH = '/content/go2_policy_export.zip'

os.makedirs(EXPORT_DIR, exist_ok=True)

# 清理旧导出，避免重复运行时目录残留。
for name in os.listdir(EXPORT_DIR):
    path = os.path.join(EXPORT_DIR, name)
    if os.path.isdir(path):
        shutil.rmtree(path)
    else:
        os.remove(path)

policy_paths = [
    ('trotting_2hz_policy', model_path),
    ('forward_locomotion_policy', model_path_forward),
]

for save_name, src in policy_paths:
    if not os.path.exists(src):
        print(f'未找到策略文件：{src}')
        continue

    dst = os.path.join(EXPORT_DIR, save_name)
    if os.path.isdir(src):
        shutil.copytree(src, dst)
    else:
        shutil.copy2(src, dst)
    print(f'已导出: {src} -> {dst}')

if os.path.exists(ZIP_PATH):
    os.remove(ZIP_PATH)

shutil.make_archive('/content/go2_policy_export', 'zip', EXPORT_DIR)
print(f'压缩包已生成: {ZIP_PATH}')
files.download(ZIP_PATH)


#### 训练后如何使用策略

最常见的用法有两种：

1. **在 notebook 里重新加载参数并继续做推理/渲染**。
2. **把导出的策略文件带到别的脚本里，再按同样的网络结构重建推理函数并加载参数**。

下面先给出一个最直接的 notebook 内使用示例：重新加载第二阶段保存的策略，然后对环境做一次 reset，输出单步动作，并渲染一个 rollout。

In [ ]:
# 使用示例：重新加载第二阶段前向运动策略，并做一次简单推理。
# 这段代码假设你已经按顺序运行完前面的环境定义与训练单元。

loaded_params = model.load_params(model_path_forward)
loaded_inference_fn = jax.jit(make_inference_fn(loaded_params))

# 重新构造一个用于演示的环境；这里仍然复用前面第二阶段训练时的 env_kwargs。
deploy_env = envs.get_environment('go2', **env_kwargs)
deploy_demo_env = envs.training.EpisodeWrapper(
    deploy_env,
    episode_length=400,
    action_repeat=1,
)

reset_key = jax.random.PRNGKey(0)
action_key = jax.random.PRNGKey(1)
state = deploy_demo_env.reset(reset_key)
action, _ = loaded_inference_fn(state.obs, action_key)

print('单步动作 shape:', action.shape)
print('前 4 维动作示例:', np.array(action[:4]))

# 再渲染一个 rollout，观察加载后的策略是否能正常工作。
render_rollout(
    jax.jit(deploy_demo_env.reset),
    jax.jit(deploy_demo_env.step),
    loaded_inference_fn,
    deploy_demo_env,
    n_steps=200,
    camera='track'
)
